In [5]:
using PowerModels, Ipopt, JuMP, DataFrames, CSV, ProgressMeter

function ACOPF_dataset_with_duals(
    case_path::String,
    existing_dataset_dir::String,
    output_dir::String
)
    
    case_name = split(basename(case_path), ".")[1]
    df_pd = CSV.read(joinpath(existing_dataset_dir, "$(case_name)_pd.csv"), DataFrame)
    df_qd = CSV.read(joinpath(existing_dataset_dir, "$(case_name)_qd.csv"), DataFrame)
    
    num_samples = nrow(df_pd)
    println("Found $(num_samples) samples in existing dataset")
    
    data = parse_file(case_path)
    
    load_bus_indices = sort([parse(Int, string(n)[3:end]) for n in names(df_pd)])
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])
    bus_indices = sort([b["bus_i"] for (i,b) in data["bus"]])
    
    optimizer = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    results = []
    
    total_time = @elapsed begin
        @showprogress "Extracting Duals" for row_idx in 1:num_samples
            temp_data = deepcopy(data)
            
            pd_values = Dict{Int, Float64}()
            qd_values = Dict{Int, Float64}()
            
            for (col_idx, bus_idx) in enumerate(load_bus_indices)
                pd_values[bus_idx] = df_pd[row_idx, col_idx]
                qd_values[bus_idx] = df_qd[row_idx, col_idx]
            end
            
            for (load_id, load_data) in temp_data["load"]
                bus_idx = load_data["load_bus"]
                if haskey(pd_values, bus_idx)
                    load_data["pd"] = pd_values[bus_idx]
                    load_data["qd"] = qd_values[bus_idx]
                end
            end
            
            try
                pm = instantiate_model(temp_data, ACPPowerModel, PowerModels.build_opf; 
                                      setting = Dict("output" => Dict("duals" => true)))
                result = optimize_model!(pm, optimizer = optimizer)
                
                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    sol = result["solution"]
                    
                    # Using JuMP to extract mu
                    mu_pg_min_vals = zeros(length(gen_indices))
                    mu_pg_max_vals = zeros(length(gen_indices))
                    mu_qg_min_vals = zeros(length(gen_indices))
                    mu_qg_max_vals = zeros(length(gen_indices))
                    mu_vm_min_vals = zeros(length(bus_indices))
                    mu_vm_max_vals = zeros(length(bus_indices))
                    
                    nw_0 = pm.var[:it][:pm][:nw][0]
                    
                    # PG duals
                    if haskey(nw_0, :pg)
                        pg_vars = nw_0[:pg]
                        for (i, gen_id) in enumerate(gen_indices)
                            try
                                pg_var = pg_vars[gen_id]
                                if has_lower_bound(pg_var)
                                    mu_pg_min_vals[i] = dual(LowerBoundRef(pg_var))
                                end
                                if has_upper_bound(pg_var)
                                    mu_pg_max_vals[i] = dual(UpperBoundRef(pg_var))
                                end
                            catch
                            end
                        end
                    end
                    
                    # QG duals
                    if haskey(nw_0, :qg)
                        qg_vars = nw_0[:qg]
                        for (i, gen_id) in enumerate(gen_indices)
                            try
                                qg_var = qg_vars[gen_id]
                                if has_lower_bound(qg_var)
                                    mu_qg_min_vals[i] = dual(LowerBoundRef(qg_var))
                                end
                                if has_upper_bound(qg_var)
                                    mu_qg_max_vals[i] = dual(UpperBoundRef(qg_var))
                                end
                            catch
                            end
                        end
                    end
                    
                    # VM duals
                    if haskey(nw_0, :vm)
                        vm_vars = nw_0[:vm]
                        for (i, bus_id) in enumerate(bus_indices)
                            try
                                vm_var = vm_vars[bus_id]
                                if has_lower_bound(vm_var)
                                    mu_vm_min_vals[i] = dual(LowerBoundRef(vm_var))
                                end
                                if has_upper_bound(vm_var)
                                    mu_vm_max_vals[i] = dual(UpperBoundRef(vm_var))
                                end
                            catch
                            end
                        end
                    end
                    
                    # Using PowerModels to extract lambda（KCL）
                    lambda_kcl_r = [get(sol["bus"][string(i)], "lam_kcl_r", 0.0) for i in bus_indices]
                    lambda_kcl_i = [get(sol["bus"][string(i)], "lam_kcl_i", 0.0) for i in bus_indices]
                    
                    push!(results, (
                        mu_pg_min=mu_pg_min_vals,
                        mu_pg_max=mu_pg_max_vals,
                        mu_qg_min=mu_qg_min_vals,
                        mu_qg_max=mu_qg_max_vals,
                        mu_vm_min=mu_vm_min_vals,
                        mu_vm_max=mu_vm_max_vals,
                        lambda_kcl_r=lambda_kcl_r,
                        lambda_kcl_i=lambda_kcl_i
                    ))
                else
                    push!(results, (
                        mu_pg_min=zeros(length(gen_indices)),
                        mu_pg_max=zeros(length(gen_indices)),
                        mu_qg_min=zeros(length(gen_indices)),
                        mu_qg_max=zeros(length(gen_indices)),
                        mu_vm_min=zeros(length(bus_indices)),
                        mu_vm_max=zeros(length(bus_indices)),
                        lambda_kcl_r=zeros(length(bus_indices)),
                        lambda_kcl_i=zeros(length(bus_indices))
                    ))
                end
            catch e
                push!(results, (
                    mu_pg_min=zeros(length(gen_indices)),
                    mu_pg_max=zeros(length(gen_indices)),
                    mu_qg_min=zeros(length(gen_indices)),
                    mu_qg_max=zeros(length(gen_indices)),
                    mu_vm_min=zeros(length(bus_indices)),
                    mu_vm_max=zeros(length(bus_indices)),
                    lambda_kcl_r=zeros(length(bus_indices)),
                    lambda_kcl_i=zeros(length(bus_indices))
                ))
            end
        end
    end
    
    println("\nProcessed $(length(results)) samples in $(round(total_time, digits=2))s")
    println("Average time per sample: $(round(total_time/length(results)*1000, digits=2))ms")
    
    mkpath(output_dir)
    
    df_mu_pg_min = DataFrame(hcat([r.mu_pg_min for r in results]...)', Symbol.("mu_pg_min_" .* string.(gen_indices)))
    df_mu_pg_max = DataFrame(hcat([r.mu_pg_max for r in results]...)', Symbol.("mu_pg_max_" .* string.(gen_indices)))
    df_mu_qg_min = DataFrame(hcat([r.mu_qg_min for r in results]...)', Symbol.("mu_qg_min_" .* string.(gen_indices)))
    df_mu_qg_max = DataFrame(hcat([r.mu_qg_max for r in results]...)', Symbol.("mu_qg_max_" .* string.(gen_indices)))
    df_mu_vm_min = DataFrame(hcat([r.mu_vm_min for r in results]...)', Symbol.("mu_vm_min_" .* string.(bus_indices)))
    df_mu_vm_max = DataFrame(hcat([r.mu_vm_max for r in results]...)', Symbol.("mu_vm_max_" .* string.(bus_indices)))
    df_lambda_r = DataFrame(hcat([r.lambda_kcl_r for r in results]...)', Symbol.("lambda_kcl_r" .* string.(bus_indices)))
    df_lambda_i = DataFrame(hcat([r.lambda_kcl_i for r in results]...)', Symbol.("lambda_kcl_i" .* string.(bus_indices)))
    
    CSV.write(joinpath(output_dir, "$(case_name)_mu_pg_min.csv"), df_mu_pg_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_pg_max.csv"), df_mu_pg_max)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_qg_min.csv"), df_mu_qg_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_qg_max.csv"), df_mu_qg_max)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_vm_min.csv"), df_mu_vm_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_vm_max.csv"), df_mu_vm_max)
    CSV.write(joinpath(output_dir, "$(case_name)_lambda_kcl_r.csv"), df_lambda_r)
    CSV.write(joinpath(output_dir, "$(case_name)_lambda_kcl_i.csv"), df_lambda_i)
    
    println("Non-zero mu_pg_min: $(sum([sum(abs.(r.mu_pg_min) .> 1e-6) for r in results]))")
    println("Non-zero mu_pg_max: $(sum([sum(abs.(r.mu_pg_max) .> 1e-6) for r in results]))")
    println("Non-zero mu_qg_min: $(sum([sum(abs.(r.mu_qg_min) .> 1e-6) for r in results]))")
    println("Non-zero mu_qg_max: $(sum([sum(abs.(r.mu_qg_max) .> 1e-6) for r in results]))")
    println("Non-zero mu_vm_min: $(sum([sum(abs.(r.mu_vm_min) .> 1e-6) for r in results]))")
    println("Non-zero mu_vm_max: $(sum([sum(abs.(r.mu_vm_max) .> 1e-6) for r in results]))")
    println("Non-zero lambda_kcl_r: $(sum([sum(abs.(r.lambda_kcl_r) .> 1e-6) for r in results]))")
    println("Non-zero lambda_kcl_i: $(sum([sum(abs.(r.lambda_kcl_i) .> 1e-6) for r in results]))")
end

function main()
    CASE_FILE = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case30_ieee.m"
    EXISTING_DATASET = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case30(v=0.12)"
    OUTPUT_DIR = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case30(v=0.25)_with_duals"
    
    ACOPF_dataset_with_duals(CASE_FILE, EXISTING_DATASET, OUTPUT_DIR)
end

main()

Found 41527 samples in existing dataset
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1842.1527999999998, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 5: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 2: [5218.2254, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 3: Float64[]


Extracting Duals 100%|███████████████████████████████████| Time: 0:29:04



Processed 41527 samples in 1744.13s
Average time per sample: 42.0ms
Non-zero mu_pg_min: 0
Non-zero mu_pg_max: 181028
Non-zero mu_qg_min: 0
Non-zero mu_qg_max: 110657
Non-zero mu_vm_min: 50114
Non-zero mu_vm_max: 716744
Non-zero lambda_kcl_r: 1245810
Non-zero lambda_kcl_i: 1107305


In [19]:
"""
ACOPF Dataset Dual Variable Extraction (V3 - with Line Flow Duals)
==================================================================
Extracts ALL dual variables needed for PINN training:

1. mu_pg_min / mu_pg_max  — Generator Pg bound duals        (n_gen each)
2. mu_qg_min / mu_qg_max  — Generator Qg bound duals        (n_gen each)
3. mu_vm_min / mu_vm_max  — Bus Vm bound duals               (n_bus each)
4. lambda_kcl_r / lambda_kcl_i — KCL equality constraint duals (n_bus each)
5. mu_sm_fr / mu_sm_to    — Branch thermal limit duals        (n_branch each)

Key insight for thermal duals:
    In PowerModels ACPPowerModel, thermal limits are unnamed quadratic constraints:
        p_fr^2 + q_fr^2 <= rate_a^2   (from side)
        p_to^2 + q_to^2 <= rate_a^2   (to side)
    They are of type: QuadExpr in MathOptInterface.LessThan{Float64}
    Total count = 2 * n_branches_with_limits
    Order: alternating [fr_1, to_1, fr_2, to_2, ...] by branch index

Usage:
    Modify the paths at the bottom, then run:
    julia acopf_constraints_with_duals_v3.jl
"""

using PowerModels, Ipopt, JuMP, MathOptInterface, DataFrames, CSV, ProgressMeter
const MOI = MathOptInterface

function ACOPF_dataset_with_duals_v3(
    case_path::String,
    existing_dataset_dir::String,
    output_dir::String
)
    case_name = split(basename(case_path), ".")[1]
    df_pd = CSV.read(joinpath(existing_dataset_dir, "$(case_name)_pd.csv"), DataFrame)
    df_qd = CSV.read(joinpath(existing_dataset_dir, "$(case_name)_qd.csv"), DataFrame)

    num_samples = nrow(df_pd)
    println("=" ^ 70)
    println("ACOPF Dual Variable Extraction V3")
    println("=" ^ 70)
    println("Case: $(case_name)")
    println("Dataset: $(existing_dataset_dir)")
    println("Output:  $(output_dir)")
    println("Samples: $(num_samples)")

    data = parse_file(case_path)

    load_bus_indices = sort([parse(Int, string(n)[3:end]) for n in names(df_pd)])
    gen_indices    = sort([g["index"]  for (i, g) in data["gen"]])
    bus_indices    = sort([b["bus_i"]  for (i, b) in data["bus"]])
    branch_indices = sort([br["index"] for (i, br) in data["branch"]])

    n_gen    = length(gen_indices)
    n_bus    = length(bus_indices)
    n_branch = length(branch_indices)

    # Count branches with thermal limits
    n_branches_with_limits = 0
    for br_id in branch_indices
        br = data["branch"][string(br_id)]
        rate_a = get(br, "rate_a", 0.0)
        if rate_a > 0 && isfinite(rate_a)
            n_branches_with_limits += 1
        end
    end
    
    expected_quad_leq = 2 * n_branches_with_limits

    println("System: $(n_bus) buses, $(n_gen) gens, $(n_branch) branches")
    println("Branches with thermal limits: $(n_branches_with_limits) / $(n_branch)")
    println("Expected QuadExpr-LessThan constraints: $(expected_quad_leq)")
    println("=" ^ 70)

    optimizer = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")

    results = []
    n_success = 0
    n_fail = 0

    # ========== Verify constraint structure on first sample ==========
    println("\nVerifying constraint structure on first sample...")
    temp_data_check = deepcopy(data)
    pm_check = instantiate_model(temp_data_check, ACPPowerModel, PowerModels.build_opf;
                                  setting = Dict("output" => Dict("duals" => true)))
    result_check = optimize_model!(pm_check, optimizer = optimizer)
    
    model_check = pm_check.model
    quad_leq_cons = all_constraints(model_check, QuadExpr, MOI.LessThan{Float64})
    actual_quad_leq = length(quad_leq_cons)
    
    println("  Actual QuadExpr-LessThan constraints: $(actual_quad_leq)")
    
    if actual_quad_leq == expected_quad_leq
        println("  ✓ Match! $(actual_quad_leq) = 2 × $(n_branches_with_limits)")
        println("  Interpretation: constraints ordered as [fr_1, to_1, fr_2, to_2, ...]")
    elseif actual_quad_leq == 0
        println("  ⚠ No quadratic inequality constraints found.")
        println("  Thermal limit duals will be all zeros.")
    else
        println("  ⚠ Mismatch: got $(actual_quad_leq), expected $(expected_quad_leq)")
        println("  Will still extract but ordering may need verification.")
    end
    
    # Show a few dual values for verification
    if actual_quad_leq > 0 && result_check["termination_status"] == MOI.LOCALLY_SOLVED
        println("\n  First 6 QuadExpr-LessThan duals (should be ≤ 0 for ≤ constraints):")
        for i in 1:min(6, actual_quad_leq)
            d = dual(quad_leq_cons[i])
            println("    [$i] dual = $(round(d, digits=8))")
        end
    end

    # ========== Main extraction loop ==========
    println("\n" * "=" ^ 70)
    println("Starting extraction...")
    println("=" ^ 70)

    total_time = @elapsed begin
        @showprogress "Extracting Duals" for row_idx in 1:num_samples
            temp_data = deepcopy(data)

            # Set load values
            pd_values = Dict{Int, Float64}()
            qd_values = Dict{Int, Float64}()
            for (col_idx, bus_idx) in enumerate(load_bus_indices)
                pd_values[bus_idx] = df_pd[row_idx, col_idx]
                qd_values[bus_idx] = df_qd[row_idx, col_idx]
            end

            for (load_id, load_data) in temp_data["load"]
                bus_idx = load_data["load_bus"]
                if haskey(pd_values, bus_idx)
                    load_data["pd"] = pd_values[bus_idx]
                    load_data["qd"] = qd_values[bus_idx]
                end
            end

            # Initialize arrays
            mu_pg_min_vals  = zeros(n_gen)
            mu_pg_max_vals  = zeros(n_gen)
            mu_qg_min_vals  = zeros(n_gen)
            mu_qg_max_vals  = zeros(n_gen)
            mu_vm_min_vals  = zeros(n_bus)
            mu_vm_max_vals  = zeros(n_bus)
            lambda_kcl_r    = zeros(n_bus)
            lambda_kcl_i    = zeros(n_bus)
            mu_sm_fr_vals   = zeros(n_branch)
            mu_sm_to_vals   = zeros(n_branch)

            try
                pm = instantiate_model(temp_data, ACPPowerModel, PowerModels.build_opf;
                                       setting = Dict("output" => Dict("duals" => true)))
                result = optimize_model!(pm, optimizer = optimizer)

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    n_success += 1
                    sol = result["solution"]
                    nw_0 = pm.var[:it][:pm][:nw][0]

                    # ========== PG duals ==========
                    if haskey(nw_0, :pg)
                        pg_vars = nw_0[:pg]
                        for (i, gen_id) in enumerate(gen_indices)
                            try
                                pg_var = pg_vars[gen_id]
                                if has_lower_bound(pg_var)
                                    mu_pg_min_vals[i] = dual(LowerBoundRef(pg_var))
                                end
                                if has_upper_bound(pg_var)
                                    mu_pg_max_vals[i] = dual(UpperBoundRef(pg_var))
                                end
                            catch; end
                        end
                    end

                    # ========== QG duals ==========
                    if haskey(nw_0, :qg)
                        qg_vars = nw_0[:qg]
                        for (i, gen_id) in enumerate(gen_indices)
                            try
                                qg_var = qg_vars[gen_id]
                                if has_lower_bound(qg_var)
                                    mu_qg_min_vals[i] = dual(LowerBoundRef(qg_var))
                                end
                                if has_upper_bound(qg_var)
                                    mu_qg_max_vals[i] = dual(UpperBoundRef(qg_var))
                                end
                            catch; end
                        end
                    end

                    # ========== VM duals ==========
                    if haskey(nw_0, :vm)
                        vm_vars = nw_0[:vm]
                        for (i, bus_id) in enumerate(bus_indices)
                            try
                                vm_var = vm_vars[bus_id]
                                if has_lower_bound(vm_var)
                                    mu_vm_min_vals[i] = dual(LowerBoundRef(vm_var))
                                end
                                if has_upper_bound(vm_var)
                                    mu_vm_max_vals[i] = dual(UpperBoundRef(vm_var))
                                end
                            catch; end
                        end
                    end

                    # ========== Lambda KCL ==========
                    lambda_kcl_r = [get(sol["bus"][string(i)], "lam_kcl_r", 0.0) for i in bus_indices]
                    lambda_kcl_i = [get(sol["bus"][string(i)], "lam_kcl_i", 0.0) for i in bus_indices]

                    # ========== Branch thermal limit duals (from JuMP model) ==========
                    if actual_quad_leq > 0
                        jmp_model = pm.model
                        quad_cons = all_constraints(jmp_model, QuadExpr, MOI.LessThan{Float64})
                        
                        # Constraints are ordered as [fr_1, to_1, fr_2, to_2, ...]
                        # where branch ordering follows PowerModels' internal order
                        # Since we have n_branches_with_limits branches, we expect
                        # 2 * n_branches_with_limits constraints
                        
                        br_idx_in_output = 0
                        for (i, br_id) in enumerate(branch_indices)
                            br = data["branch"][string(br_id)]
                            rate_a = get(br, "rate_a", 0.0)
                            if rate_a > 0 && isfinite(rate_a)
                                br_idx_in_output += 1
                                con_idx_fr = 2 * br_idx_in_output - 1
                                con_idx_to = 2 * br_idx_in_output
                                
                                if con_idx_fr <= length(quad_cons)
                                    try
                                        mu_sm_fr_vals[i] = dual(quad_cons[con_idx_fr])
                                    catch; end
                                end
                                if con_idx_to <= length(quad_cons)
                                    try
                                        mu_sm_to_vals[i] = dual(quad_cons[con_idx_to])
                                    catch; end
                                end
                            end
                            # Branches without rate_a keep mu = 0
                        end
                    end
                else
                    n_fail += 1
                end
            catch e
                n_fail += 1
            end

            push!(results, (
                mu_pg_min    = mu_pg_min_vals,
                mu_pg_max    = mu_pg_max_vals,
                mu_qg_min    = mu_qg_min_vals,
                mu_qg_max    = mu_qg_max_vals,
                mu_vm_min    = mu_vm_min_vals,
                mu_vm_max    = mu_vm_max_vals,
                lambda_kcl_r = lambda_kcl_r,
                lambda_kcl_i = lambda_kcl_i,
                mu_sm_fr     = mu_sm_fr_vals,
                mu_sm_to     = mu_sm_to_vals
            ))
        end
    end

    # ========== Save results ==========
    println("\n" * "=" ^ 70)
    println("Results Summary")
    println("=" ^ 70)
    println("Processed $(length(results)) samples in $(round(total_time, digits=2))s")
    println("  Success: $(n_success), Failed: $(n_fail)")
    println("Average time per sample: $(round(total_time/length(results)*1000, digits=2))ms")

    mkpath(output_dir)

    df_mu_pg_min = DataFrame(hcat([r.mu_pg_min for r in results]...)', Symbol.("mu_pg_min_" .* string.(gen_indices)))
    df_mu_pg_max = DataFrame(hcat([r.mu_pg_max for r in results]...)', Symbol.("mu_pg_max_" .* string.(gen_indices)))
    df_mu_qg_min = DataFrame(hcat([r.mu_qg_min for r in results]...)', Symbol.("mu_qg_min_" .* string.(gen_indices)))
    df_mu_qg_max = DataFrame(hcat([r.mu_qg_max for r in results]...)', Symbol.("mu_qg_max_" .* string.(gen_indices)))
    df_mu_vm_min = DataFrame(hcat([r.mu_vm_min for r in results]...)', Symbol.("mu_vm_min_" .* string.(bus_indices)))
    df_mu_vm_max = DataFrame(hcat([r.mu_vm_max for r in results]...)', Symbol.("mu_vm_max_" .* string.(bus_indices)))
    df_lambda_r  = DataFrame(hcat([r.lambda_kcl_r for r in results]...)', Symbol.("lambda_kcl_r_" .* string.(bus_indices)))
    df_lambda_i  = DataFrame(hcat([r.lambda_kcl_i for r in results]...)', Symbol.("lambda_kcl_i_" .* string.(bus_indices)))
    df_mu_sm_fr  = DataFrame(hcat([r.mu_sm_fr for r in results]...)', Symbol.("mu_sm_fr_" .* string.(branch_indices)))
    df_mu_sm_to  = DataFrame(hcat([r.mu_sm_to for r in results]...)', Symbol.("mu_sm_to_" .* string.(branch_indices)))

    CSV.write(joinpath(output_dir, "$(case_name)_mu_pg_min.csv"), df_mu_pg_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_pg_max.csv"), df_mu_pg_max)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_qg_min.csv"), df_mu_qg_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_qg_max.csv"), df_mu_qg_max)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_vm_min.csv"), df_mu_vm_min)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_vm_max.csv"), df_mu_vm_max)
    CSV.write(joinpath(output_dir, "$(case_name)_lambda_kcl_r.csv"), df_lambda_r)
    CSV.write(joinpath(output_dir, "$(case_name)_lambda_kcl_i.csv"), df_lambda_i)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_sm_fr.csv"), df_mu_sm_fr)
    CSV.write(joinpath(output_dir, "$(case_name)_mu_sm_to.csv"), df_mu_sm_to)

    println("\nDual variable statistics (non-zero counts):")
    println("  mu_pg_min:    $(sum([sum(abs.(r.mu_pg_min) .> 1e-6) for r in results]))")
    println("  mu_pg_max:    $(sum([sum(abs.(r.mu_pg_max) .> 1e-6) for r in results]))")
    println("  mu_qg_min:    $(sum([sum(abs.(r.mu_qg_min) .> 1e-6) for r in results]))")
    println("  mu_qg_max:    $(sum([sum(abs.(r.mu_qg_max) .> 1e-6) for r in results]))")
    println("  mu_vm_min:    $(sum([sum(abs.(r.mu_vm_min) .> 1e-6) for r in results]))")
    println("  mu_vm_max:    $(sum([sum(abs.(r.mu_vm_max) .> 1e-6) for r in results]))")
    println("  lambda_kcl_r: $(sum([sum(abs.(r.lambda_kcl_r) .> 1e-6) for r in results]))")
    println("  lambda_kcl_i: $(sum([sum(abs.(r.lambda_kcl_i) .> 1e-6) for r in results]))")
    println("  mu_sm_fr:     $(sum([sum(abs.(r.mu_sm_fr) .> 1e-6) for r in results]))")
    println("  mu_sm_to:     $(sum([sum(abs.(r.mu_sm_to) .> 1e-6) for r in results]))")

    println("\nFiles saved to: $(output_dir)")
    println("=" ^ 70)
end


# =====================================================================
# Main - Modify paths below before running
# =====================================================================
function main()
    CASE_FILE         = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case300_ieee.m"
    EXISTING_DATASET  = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)"
    OUTPUT_DIR        = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)_with_duals"

    ACOPF_dataset_with_duals_v3(CASE_FILE, EXISTING_DATASET, OUTPUT_DIR)
end

main()

ACOPF Dual Variable Extraction V3
Case: pglib_opf_case300_ieee
Dataset: C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)
Output:  C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)_with_duals
Samples: 16146
[info | PowerModels]: removing 1 cost terms from generator 32: [3101.6664, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 29: [2727.8538000000003, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 1: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 54: [10398.7496, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 41: [3965.7688, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 65: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 51: [3512.6651, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 53: [2999.2014999999997, 0.0]
[info | PowerModels]: removing 1 cost terms from genera

Extracting Duals 100%|███████████████████████████████████| Time: 3:09:47m



Results Summary
Processed 16146 samples in 11387.34s
  Success: 16146, Failed: 0
Average time per sample: 705.27ms

Dual variable statistics (non-zero counts):
  mu_pg_min:    205820
  mu_pg_max:    622782
  mu_qg_min:    70208
  mu_qg_max:    613661
  mu_vm_min:    812930
  mu_vm_max:    3505260
  lambda_kcl_r: 4843800
  lambda_kcl_i: 4384709
  mu_sm_fr:     510682
  mu_sm_to:     492489

Files saved to: C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)_with_duals


In [9]:
"""
Diagnostic v2: Find thermal limit constraints directly from JuMP model
"""

using PowerModels, Ipopt, JuMP, MathOptInterface
const MOI = MathOptInterface

function diagnose_v2(case_path::String)
    data = parse_file(case_path)
    
    optimizer = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    pm = instantiate_model(data, ACPPowerModel, PowerModels.build_opf;
                           setting = Dict("output" => Dict("duals" => true)))
    result = optimize_model!(pm, optimizer = optimizer)
    
    println("Status: $(result["termination_status"])")
    
    # Get the underlying JuMP model
    model = pm.model
    
    # ========== List ALL constraint types ==========
    println("\n" * "=" ^ 70)
    println("All constraint types in JuMP model:")
    println("=" ^ 70)
    
    con_types = list_of_constraint_types(model)
    for (F, S) in con_types
        n = num_constraints(model, F, S)
        println("  $(F) in $(S): $(n) constraints")
        
        # Show first constraint and its dual
        cons = all_constraints(model, F, S)
        if length(cons) > 0
            c = cons[1]
            try
                d = dual(c)
                println("    First constraint name: $(name(c))")
                println("    First dual value: $(d)")
            catch e
                println("    First constraint name: $(name(c))")
                println("    Dual error: $(e)")
            end
        end
    end
    
    # ========== Find quadratic constraints (thermal limits are quadratic) ==========
    println("\n" * "=" ^ 70)
    println("Looking for quadratic constraints (likely thermal limits):")
    println("=" ^ 70)
    
    # In ACPPowerModel, thermal limits are:  p^2 + q^2 <= rate_a^2
    # This is ScalarQuadraticFunction-in-LessThan
    
    for (F, S) in con_types
        if occursin("Quadratic", string(F)) || occursin("LessThan", string(S))
            cons = all_constraints(model, F, S)
            println("\n  $(F) in $(S): $(length(cons)) constraints")
            
            # Show first few constraint names
            for (i, c) in enumerate(cons)
                if i <= 10
                    cname = name(c)
                    try
                        d = dual(c)
                        println("    [$i] name='$(cname)', dual=$(round(d, digits=6))")
                    catch
                        println("    [$i] name='$(cname)', dual=N/A")
                    end
                end
            end
            if length(cons) > 10
                println("    ... ($(length(cons) - 10) more)")
            end
        end
    end
    
    # ========== Try to find constraints by name pattern ==========
    println("\n" * "=" ^ 70)
    println("Searching all named constraints for 'thermal', 'flow', 'sm', 'rate':")
    println("=" ^ 70)
    
    found = 0
    for (F, S) in con_types
        cons = all_constraints(model, F, S)
        for c in cons
            cname = string(name(c))
            if any(s -> occursin(s, lowercase(cname)), ["thermal", "flow", "sm_", "rate", "branch"])
                try
                    d = dual(c)
                    println("  FOUND: '$(cname)' → dual = $(round(d, digits=6))")
                catch
                    println("  FOUND: '$(cname)' → dual = N/A")
                end
                found += 1
                if found >= 20
                    println("  ... (stopping after 20 matches)")
                    break
                end
            end
        end
        if found >= 20
            break
        end
    end
    
    if found == 0
        println("  No constraints found matching these patterns.")
        println("\n  Listing ALL named constraints (first 30):")
        count = 0
        for (F, S) in con_types
            cons = all_constraints(model, F, S)
            for c in cons
                cname = name(c)
                if cname != ""
                    println("    '$(cname)'")
                    count += 1
                    if count >= 30
                        break
                    end
                end
            end
            if count >= 30
                break
            end
        end
    end
    
    println("\n" * "=" ^ 70)
    println("Diagnosis v2 complete")
    println("=" ^ 70)
end

CASE_FILE = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case30_ieee.m"
diagnose_v2(CASE_FILE)

[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1842.1527999999998, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 5: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 2: [5218.2254, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 3: Float64[]
Status: LOCALLY_SOLVED

All constraint types in JuMP model:
  NonlinearExpr in MathOptInterface.EqualTo{Float64}: 164 constraints
    First constraint name: 
    First dual value: 5218.225400452016
  AffExpr in MathOptInterface.EqualTo{Float64}: 57 constraints
    First constraint name: 
    First dual value: 8.51963470244013e-12
  AffExpr in MathOptInterface.Interval{Float64}: 41 constraints
    First constraint name: 
    First dual value: -5.983443582482624e-8
  QuadExpr in MathOptInterface.EqualTo{Float64}: 4 constraints
    First cons